In [ ]:
# IMPORTS

import torch
import torch.optim.lr_scheduler as lr_scheduler
from sklearn.model_selection import train_test_split

# simulation data saving
import csv
import json
import os
import sys

#
import numpy as np

# custom functions
from functions import import_imagedata, ImageProcessor, label_oh_tf
from fns4wandb import  set_lossfn
from architectures import sevennet, smallnet1, smallnet2, smallnet3
from loop_fns import loop

# print presentation
import collections
import pprint
pp = pprint.PrettyPrinter(indent=4)
from datetime import date
d = date.today()
print(str(d), type(str(d))) # check date
from tqdm import tqdm


# save data
from datetime import date
import collections
import time
# save data format
import csv
import json
import pickle
import os

# Simulation Manager
import wandb 

from fns4wandb import choose_model, set_lossfn


In [ ]:
# file paths
_localDir = "../../" # replace with your own directory
_save_location = _localDir+r'antvis/optics/res_big_loop_saves/'

data_path = _localDir+r'/antvis/optics/AugmentedDS_IDSW/'

gitHASH = '5b835027b98560be2802b81891d000be97a6c8bc'

In [ ]:
wandb.login()

Maths to calc percentage coverage of image at different resolutions by 5 pixels.
5 pixels was chosen as the yaw padding (left/right copy&swap)

452 144               5/452 *100 = 1%
226 72                5/226 *100 = 2%
113 36                5/113 *100 = 4% -- 2/113 *100= 1.7% ~ 2%
57  18   (56.5,)      5/57 *100  = 8% -- 2/57 *100 = 3.5% ~ 4%.   1/57 = 1.75%
29   9   (28.5,)      5/29 *100  = 17% -- 2/29 *100 = 6.89 ~ 7%   1/28 = 3.57 ~ 4%
15   5   (14.5, 4.5)
8    3   (7.5,2.5)
4,   2   (, 1.5)

In [ ]:
# save to csv 

def save2csv_nest_dict(nested_dict, file_name, save_location:str):
  """ Saves a nested dictionary in a csv format.
      Takes a nested dictionary object
      Writes dara into a human-readible csv
      The forloops retreive data from within the nested dict structre."""
  
  # flattern nested dictionary
  flatterend_dict = {}
  for k,v in nested_dict.items():
    if isinstance(v, dict):
      for nested_key, nested_val in v.items():
        flatterend_dict[f"{k}_{nested_key}"] = nested_val
    else:
      flatterend_dict[k] =v

  columns = list(flatterend_dict.keys())

  with open(save_location+str(file_name)+'.csv', "a+", newline="") as f:
      # using dictwriter
      writer = csv.DictWriter(f, fieldnames=columns)
      # using writeheader function
      if f.tell() == 0:
        writer.writeheader()
      writer.writerow(flatterend_dict)
      f.close()


def check_obj4np(obj):
    """
    Takes nested data object
    returns data (array or tensor) as a list"""
    if isinstance(obj, dict):
        return {key: check_obj4np(value) for key, value in obj.items()}
    if isinstance(obj,list):
        return [check_obj4np(item) for item in obj]
    if isinstance(obj,np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.tolist()
    else:
        return obj


def save2josn_nested_dict(nested_dict, file_name, save_location:str):
    """
    Takes a nested dictionary object
    Writes a json file
    Uses check_obj4np for data format conversion
    """
    nested_dict = check_obj4np(nested_dict)
    json_obj = json.dumps(nested_dict, indent=4)
    with open(save_location+str(file_name)+'.json', 'a+') as f:
        f.write(json_obj)
        f.close()


def save2csv(nested_dict, file_name, save_location:str):
    """
    Takes a nested dictionary object
    Uses check_obj4np for data format conversion
    Writes data into a human-readible csv
    """
    nested_dict = check_obj4np(nested_dict)
    
    columns = list(nested_dict.keys())
    path = os.path.join(save_location, file_name +".csv")
    try:
        with open(path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            # using dictwriter
            # using writeheader function
            if f.tell() == 0:
                writer.writeheader()
            writer.writerow(nested_dict)
            f.close()
    except IOError as e:
        print("I/O error({0}): {1}".format(e.errno, e.strerror))
    except ValueError:
              print("could not convert to string")
    except:
              print("unexpected error: ", sys.exc_info()[0])
        

def save2json(nested_dict, file_name, save_location:str):
    """
    Takes a nested dictionary object
    Uses check_obj4np for data format conversion
    Writes a json file of extracted data.
    """
    nested_dict = check_obj4np(nested_dict)
    json_obj = json.dumps(nested_dict, indent=4)
    path = os.path.join(save_location, file_name+".json")
    with open(path, 'w') as f:
        f.write(json_obj)
        

def read_in_json(file_path, file_name):
    """
    Read in json file
    """
    path = os.path.join(file_path, 'file_name')
    try:
        with open(path, 'r') as f:
            #obj = f.read()
            dj = json.load(f, object_pairs_hook= collections.OrderedDict) #obj, 
            print(dj)
    except Exception as e:
        print("Error decoding Json")
        print(e)


'\ndef save2csv(nested_dict, file_name, save_location:str):\n  # flattern nested dictionary\n  #flatterend_dict = {}\n  #for k,v in nested_dict.items():\n  #  if isinstance(v, dict):\n  #    for nested_key, nested_val in v.items():\n  #      flatterend_dict[f"{k}_{nested_key}"] = nested_val\n  #  else:\n  #    flatterend_dict[k] =v\n\n  columns = list(nested_dict.keys())\n\n  with open(save_location+str(file_name)+\'.csv\', "a", newline="") as f:\n      # using dictwriter\n      writer = csv.DictWriter(f, fieldnames=columns)\n      # using writeheader function\n      #if f.tell() == 0:\n      #  writer.writeheader()\n      writer.writerow(nested_dict)\n      f.close()\n'

In [ ]:
# dictionaries holding the first linear layer sizes for each model for multiple image resolutions

date = date.today()

model_card_vgg = {'name': 'vgg'+str(date), 'model': 'vgg16',
                  'f_lin_lay':[4096,
                             4096,
                             4096,
                             4096,
                             4096,
                             4096,
                             4096,
                            ]},


model_card_7c3l = {'name': '7c3l'+str(date), 'model': '7c3l', 'channels': 3, 'ks': (3,5),
                  'f_lin_lay':[248832,    # 452 144    # p5
                            59904,      # 226 72     # p5
                            11264,      # 113  36    # p2
                            1536,       # 57 18      # p1
                            ]}



model_card_4c3l = {'name': '4c3l'+str(date), 'model': '4c3l', 'channels': 3, 'ks': (3,5),
                  'f_lin_lay':[539904,    # 452 144    # p5
                             141056,    # 226 72     # p5
                             304640,     # 113  36    # p2
                             9984,      # 57 18      # p1
                             2304,      # 29  9
                             512,       # 15 5
                             256]}      # 8 3

model_card_3c2l = {'name': '3c2l'+str(date), 'model': '3c2l', 'channels': 3, 'ks': (3,5),
                  'f_lin_lay':[1069888,    # 452 144    # p5
                             274688,     #226 72      # p5
                             68096,      # 113  36    # p2
                             17280,      # 57 18      # p1
                             3840,       # 29  9
                             960,        # 15 5
                             256]}       # 8 3

model_card_2c2l = {'name': '2c2l'+str(date), 'model': '2c2l', 'channels': 3, 'ks': (3,5),
                  'f_lin_lay':[1055232,    # 452 144    # p5
                             267264,     #226 72      # p5
                             64512,      # 113  36    # p2
                             15552,      # 57 18      # p1
                             3072,       # 29  9
                             640,        # 15 5
                             128]}       # 8 3

# Creation of resolution cards - these variables hold model parameter values 
# that are specific to each resolution
resolution_card_452144 = {'resolution':[452,144], 'padding':5, 'index':0}
resolution_card_22672 = {'resolution':[226,72], 'padding':5, 'index':1}
resolution_card_11336 = {'resolution':[113,36], 'padding':2, 'index':2}
resolution_card_5715 = {'resolution':[57,18], 'padding':1, 'index':3}
resolution_card_299 = {'resolution':[29,9], 'padding':0, 'index':4}
resolution_card_155 = {'resolution':[15,5], 'padding':0, 'index':5}
resolution_card_83 = {'resolution':[8,3], 'padding':0, 'index':6}

def get_lin_lay(model_card, resolution):
    if resolution == [452, 144]:
        lin_lay = model_card['f_lin_lay'][0]
    elif resolution == [226, 72]:
        lin_lay = model_card['f_lin_lay'][1]
    elif resolution == [113, 36]:
        lin_lay = model_card['f_lin_lay'][2]
    elif resolution == [57, 18]:
        lin_lay = model_card['f_lin_lay'][3]
    elif resolution == [29, 9]:
        lin_lay = model_card['f_lin_lay'][4]
    elif resolution == [15, 5]:
        lin_lay = model_card['f_lin_lay'][5]
    elif resolution == [8, 3]:
        lin_lay = model_card['f_lin_lay'][6]
    else:
        print("PARAMETER NOT FOUND: \n f_lin_lay FROM MODEL CARD")
    return lin_lay

resolution_cards = [resolution_card_452144, resolution_card_22672, resolution_card_11336, resolution_card_5715,
                    resolution_card_299, resolution_card_155, resolution_card_83] # one list for easy looping

# Simualtion parameters
learning_rate_cards = [1e-5, 1e-4, 1e-3, 1e-2, 1e-6] # learning rate
wd_cards = [4e-5, 5e-5, 1e-5, 1e-4, 5e-4, 4e-4] # weight decay
scheduler_cards = [0.2, 0.1, 0.3, 0.05, 0.001] # The value passed into LR scheduler if used
seeds = [1,3,5,7,9,34,57,79] # random seed

# model cards en listed for easy looping
model_cards =[model_card_vgg, model_card_7c3l, model_card_4c3l, model_card_3c2l, model_card_2c2l]

loss_fn_cards = ['MSE', 'CrossEntropy'] # loss function chosen for this simulation



In [ ]:
# WandB config dictionary. used to hold simulation parameter values.
config = dict({'parameters': 'parameters for big loop run'})
config.update({'model_cards':model_cards})
config.update({'resolution_cards':resolution_cards})
config.update({'learning_rate_cards':learning_rate_cards})
config.update({'wd_cards':wd_cards})
config.update({'scheduler_cards':scheduler_cards})
config.update({'seeds':seeds})
config.update({'loss_fn_cards': loss_fn_cards})

In [ ]:

def train_model(model, x_train, x_val, y_train, y_val, res, pad, save_dict, lr, scheduler, loss_fn, epochs, model_name): #train_dl, val_dl, 
    """
    This function controls the Train and Val looping for X number of epochs.
    Model and data groups are pre defined
    Parameters such as resolution (res), padding value (pad), learning rate (lr), schedular value (scheduler)
      and number of epochs of training (epochs) are passed in.
    Save_dict contains simulation result variables such as accuracy scores and test labels and test predictions.
    """

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)#build_optimizer(mo
    scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=scheduler, last_epoch=-1) 

    #model = model.to(device)
    model.train()
    

    t_loss_list = []
    v_loss_list = []
    t_predict_list = []
    v_predict_list = []
    t_accuracy_list = []
    v_accuracy_list = []
    t_label_list = []
    v_label_list = []
    total_epochs = 0
    
    for epoch in tqdm(range(epochs)):
        # loop through epochs. use of tqdm adds a nice progress bar
        if epoch == 0:
            best_model = model # best_model will be updated to hold the best performing model so far.
            best_acc = 0

        print('Training...')
        # traing loop
        t_loss, train_prediction, t_correct, model, optimizer = loop(model=model, X=x_train, Y=y_train, loss_fn=loss_fn, device=device, 
                                                                     size=res, pad=pad, num_classes=11, model_name=model_name, 
                                                                     optimizer=optimizer, scheduler=scheduler)
        # calculate accuracy as a percentage and log
        t_accuracy = (t_correct/len(x_train)*100)
        t_accuracy_list.append(t_accuracy)
        wandb.log({'t_accuracy', t_accuracy}) 

        # format and log other useful variables
        t_loss_list.append(t_loss)
        t_predict_list.append(train_prediction)
        wandb.log({'t_loss_list':t_loss})
        wandb.log({'t_predict_list':train_prediction})
        wandb.log({'train_labels':list(y_train)})
        train_acc = (t_correct / len(x_train))
        t_accuracy_list.append(train_acc)
        wandb.log({'train_acc':train_acc})
        #y_train_save = y_train.tolist()


        print('validating...')
        # validation loop
        v_loss, val_prediction, val_correct= loop(model=model, X=x_val, Y=y_val, loss_fn=loss_fn,
                                                   device=device, size=res,  pad=pad, num_classes=11,
                                                     model_name= model_name, train=False) # train=False puts the model in eval mode

        # format and log useful simulation variables
        v_loss_list.append(v_loss)
        v_predict_list.append(val_prediction)
        wandb.log({'v_loss_list':v_loss})
        v_accuracy_list.append((val_correct/len(y_val)*100))
        val_acc = (t_correct / len(x_val))
        v_accuracy_list.append(val_acc)
        wandb.log({'val_acc':(val_correct/len(y_val)*100)})
            
        total_epochs += 1 # at end of epoch, total_epochs is increased.
        

    # format and log simulation variables.
    save_dict['training_samples'] = len(x_train)
    save_dict['validation_samples'] = len(x_val)
    save_dict['Current_Epoch'] += epochs
    model = best_model
    save_dict['t_loss_list'] = t_loss_list
    save_dict['t_labels'] = y_train
    save_dict['t_predict_list'] = t_predict_list 

    #wandb.log({'y_train':y_train})
    wandb.log({'t_predict_list':t_predict_list})

    save_dict['t_accuracy_list'] = t_accuracy_list 
    #y_val_save = y_val.tolist()
    save_dict['v_loss_list'] = v_loss_list
    save_dict['v_predict_list'] = v_predict_list  #
    #save_dict['v_labels'] = y_val
    save_dict['v_accuracy_list'] = v_accuracy_list  #

    return model, save_dict

def test_loop(model, model_name, X, Y, res, pad, save_dict, loss_fn, device, num_classes=11):
    # test_loop is slightly different to the loop function used for training and testing
    model = model.eval()
    predict_list = []
    current_loss = 0
    total_count =0
    num_correct = 0
    correct = 0
    colour ='colour'
    size =  res

    with torch.no_grad():
        print('Testing...') 
        for idx, img in enumerate(X):
            prepro = ImageProcessor(device)
            if model_name == 'vgg16':
                tense = prepro.colour_size_tense(img, colour, size, pad, vg=True) # vg parameter controls whether blank boarder padding is added to image to increase image size
            elif (model_name == '7c3l' and size == [29, 9]) or (model_name == '7c3l' and size == [15, 5]) or (model_name == '7c3l' and size ==[8, 3]):
                tense = prepro.colour_size_tense(img, colour, size, pad, vg=True)
            else:
                tense = prepro.colour_size_tense(img, colour, size, pad)
            prediction = model.forward(tense)
            label = label_oh_tf(Y[idx], num_classes).to(device)
            loss = loss_fn(prediction, label)

            if prediction.argmax()==label.argmax():
                num_correct +=1
            total_count +=1
            correct +=(prediction.argmax()==label.argmax()).sum().item()

        acc = num_correct/total_count
        accuracy = 100*(acc)
        predict_list.append(prediction)
        current_loss += loss.item()
    return accuracy, predict_list, Y, current_loss


def get_data(random_seed):
    # function to handle data splitting
    # returns data groups
    file_path =  data_path
    img_len = len(os.listdir(file_path))
    
    x, y = import_imagedata(file_path)
    x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.3, train_size=0.7,
                                     random_state=random_seed, shuffle=True)
    x_train, x_val, y_train, y_val = train_test_split(x_train,y_train, test_size=0.3, train_size=0.7,
                                     random_state=random_seed, shuffle=True)

    return x_train, y_train, x_val, y_val, x_test, y_test


    
def get_lin_lay(model_card, resolution):
    # handles retriving the correct value for the first linear layer in the model head
    # each model:resolution combo requires a specific value as these two variable affect the feature map size
    if resolution == [452, 144]:
        lin_lay = model_card['f_lin_lay'][0]
    elif resolution == [226, 72]:
        lin_lay = model_card['f_lin_lay'][1]
    elif resolution == [113, 36]:
        lin_lay = model_card['f_lin_lay'][2]
    elif resolution == [57, 18]:
        lin_lay = model_card['f_lin_lay'][3]
    elif resolution == [29, 9]:
        lin_lay = model_card['f_lin_lay'][4]
    elif resolution == [15, 5]:
        lin_lay = model_card['f_lin_lay'][5]
    elif resolution == [8, 3]:
        lin_lay = model_card['f_lin_lay'][6]
    else:
        print("PARAMETER NOT FOUND: \n f_lin_lay FROM MODEL CARD")
    return lin_lay


In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"


def _go(config=None):
    """
    Handles looping through mutliple hyperparameter values 
    # takes in the config dictionary -> a requirement of the WandB simulation manager
    """
    with wandb.init(config=config, project=f"test Run Big Loop", notes="First logging test of big loop. limitied data",):
        if len(gitHASH) <1:
            print("YOU FORGET THE GIT HASH")
            return
        else:
            print('Git Hash registered')
            
        config = wandb.config
        start = time.process_time()

        for model_idx, model_card in enumerate(config.model_cards): # loop through different models
            
            model_name = model_card['model']
            model_index = model_card['idx']
            dropout = model_card['dropout'] 
            
            
            for res_idx, resolution_card in enumerate(config['resolution_cards']): # loop through different resolutions

                resolution = resolution_card['resolution']
                pad = resolution_card['padding']
               
                lin_lay = get_lin_lay(model_card, resolution)

                
                for lr_idx, lr in enumerate(config['learning_rate_cards']): # loop through different learning rates
                    for wd_idx, wd_card in enumerate(wd_cards): # loop through different weight decay values
                        for sched_idx, scheduler in enumerate(config['scheduler_cards']): # loop through different learning rate scheduler values
                            for seed_idx, seed in enumerate(config['seeds']): # loop over multiple seeds
                                seed = seed
                                for lossfn_idx, lossfn in enumerate(config['loss_fn_cards']): # loop through loss functions

                                    # Prints for human eyes -> to know what simulation hyperparameters were selected
                                    print('Model: ', str(model_name), f" idx: {model_idx} / {len(config['model_cards'])}")
                                    print('resolution: ', str(resolution), f"  idx: {res_idx} / {len(config['resolution_cards'])}")
                                    print('learning rate: ', str(lr), f"  idx: {lr_idx} / {len(config['learning_rate_cards'])}")
                                    print('weight decay: ', str(wd_card), f"  idx: {wd_idx} / {len(config['wd_cards'])}")
                                    print('scheduler: ', str(scheduler), f"  idx: {sched_idx} / {len(config['scheduler_cards'])}")
                                    print('seed: ', str(seed), f"  idx:  {seed_idx} / {len(config['seeds'])}")
                                    print('loss function: ', str(lossfn), f" idx: {lossfn_idx} / {len(config['loss_fn_cards'])}")
                                    run_start_time = time.process_time()
                                    print(run_start_time)
   
                                    print(time.process_time() - start)
                                    
                                    epochs = 2 # set here to 2 for testing

                                    print(f"Total epochs: {epochs}")
                                     
                                    wandb.log({'gitHash':gitHASH})
                                    wandb.log({'Epochs': epochs})
                                    # set save dictionary
                                    save_dict = {'Run' : f"GettingComputationTime_{model_name}{resolution}{date}",
                                                 'Current_Epoch': 0,
                                                 'save_location' : _save_location}
                                    
                                    # set model
                                    model = choose_model(model_name, lin_lay, dropout).to(device) # retreive model
                                    # get image data
                                    x_train, y_train, x_val, y_val, x_test, y_test = get_data(seed) # retreive data groups
                                    
                                    # set loss function
                                    loss_fn = set_lossfn(lossfn)
                                    
                                    # set optimizer
                                    optimizer = torch.optim.Adam(model.parameters(),lr=lr)

                                    wandb.watch(model, loss_fn, log='all', log_freq=2, idx = model_index) # set frequency of WandB logging

                                    # training and validation for X number of epochs (specified above)
                                    model, save_dict = train_model(model, x_train, x_val, y_train, y_val, resolution, pad, save_dict, lr, scheduler, loss_fn,epochs, model_name)

                                    # test                                                       
                                    test_acc,test_predict_list, y_test, test_loss = test_loop(model, model_name, x_test, y_test,resolution, pad, save_dict,loss_fn, device, num_classes=11) #model, model_name, X, Y, res, pad, loss_fn, device, num_classes=11
                                    # log test results
                                    print(' \n train acc: ', save_dict['t_accuracy_list'][-1])
                                    print(' \n val acc: ', save_dict['v_accuracy_list'][-1])
                                    print(' \n test acc: ', test_acc)
                                    # Update and log
                                    save_dict.update({'test_acc': test_acc})
                                    save_dict.update({'test_predict': test_predict_list})
                                    save_dict.update({'test_labels': list(y_test)})
                                    save_dict.update({'test_loss':test_loss})
                                    wandb.log({'test_acc': test_acc})
                                    wandb.log({'test_predict': test_predict_list})
                                    wandb.log({'test_labels': list(y_test)})

                                    #saving
                                    diction = {}
                                    d = date.today()
                                    d=str(d)
                                    diction.update({'Date':d})
                                    diction.update({'gitHASH':str(gitHASH)})
                                    diction.update({'model_name': str(model_name)})
                                    diction.update({'loss_fn': str(lossfn)})
                                    diction.update({'lr': str(lr)})
                                    diction.update({'wd': str(wd_card)})
                                    diction.update({'scheduler': str(scheduler)})
                                    diction.update({'seed': str(seed)})
                                    diction.update({'resolution': str(resolution)})
                                    diction.update({'pad': int(pad)})
                                    diction.update({'lin_lay': int(lin_lay)})
                                    diction.update({'run time': (time.process_time() - run_start_time)})
                                    diction.update(save_dict)
                                    
                                    save_location = save_dict['save_location']
                                    title = save_dict['Run']
                                    save2json(diction, title, save_location)
                                    save2csv(diction, title, save_location)
        
                                    diction['model.state_dict'] = model.state_dict() #to('cpu').
                                    # save all simulation data in pickle file
                                    with open(f"{save_location}{title}.pkl", 'wb+') as f:
                                        pickle.dump(diction, f)
                                                                        
                                    print(f' \n END {model_name} {resolution} Run Time: ',time.process_time() - run_start_time)

        print('Final Run time:  ',time.process_time() - start)

In [ ]:
_go(config)